# 1. Scenario & Problem Definition

**Objective**: Predicting the success of Kickstarter campaigns based on their initial characteristics (description, goal, category, launch time) to help creators optimize their projects before launch.

**Context**: Crowdfunding is a competitive space. A predictive model can identify "weak" projects and suggest improvements (e.g., "Goal is too high for this category", "Description lacks emotion").

**Task**: Binary Classification (Successful vs. Failed/Canceled).

---

# 2. Dataset Analysis (Raw Data)

In this notebook, we analyze the **raw** dataset to identify data quality issues such as missing values, outliers, and redundant features. These findings dictate the logic implemented in `src/preprocessing.py`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from app.src import config

# Set Plot Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Update: Save images to unified results directory
SAVE_DIR = config.RESULTS_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load Raw Data
df = pd.read_csv(config.RAW_DATA_PATH)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2.1 Missing Values Analysis
We need to identify columns with missing data to decide between dropping rows, imputing with median/mode, or filling with a placeholder.

In [ ]:
# Visualizing Missing Data
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.savefig(os.path.join(SAVE_DIR, 'missing_values_heatmap.png'))
plt.show()

In [ ]:
# Percentage of Missing Values
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing Values (%):")
print(missing)

# Decision Log:
# - blurb/name (~0.1%): Fill with empty string.
# - location/category: Low missing, can use 'Unknown' or Drop. 
# - friends/is_backing: High missing (likely PII/irrelevant), DROP.

## 2.2 Outlier Analysis
We investigate numerical features like `goal` and `duration` for extreme values that could skew the model.

In [ ]:
df['goal'] = pd.to_numeric(df['goal'], errors='coerce')
df['launched_at'] = pd.to_datetime(df['launched_at'], unit='s')
df['deadline'] = pd.to_datetime(df['deadline'], unit='s')
df['duration_days'] = (df['deadline'] - df['launched_at']).dt.days

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Goal Outliers
sns.boxplot(x=df['goal'], ax=ax[0])
ax[0].set_title('Goal Distribution (Raw)')
ax[0].set_xscale('log') # Log scale because of extreme outliers

# Duration Outliers
sns.boxplot(x=df['duration_days'], ax=ax[1])
ax[1].set_title('Duration Distribution (Days)')

plt.savefig(os.path.join(SAVE_DIR, 'outliers_distributions.png'))
plt.show()

**Observations & Strategy**:
- **Goal**: Some projects ask for $100M+ (likely jokes/errors). Strategy: Apply `np.log1p` transformation to compress scale.
- **Duration**: Most projects are 30-60 days. Anything >60 or <1 is suspicious. Strategy: Clip or Drop campaigns > 60 days.

## 2.3 Feature Redundancy (Correlation)
Checking for highly correlated features to avoid multicollinearity.

In [ ]:
# Select numeric columns
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.savefig(os.path.join(SAVE_DIR, 'correlation_matrix.png'))
plt.show()

**Decision**:
- `goal` and `goal_usd` are highly correlated. We will keep `goal_usd` (standardized) and drop `goal`.
- `pledged` is a "Leakage" feature (unknown at launch). **MUST DROP**.

# 3. Execution (Apply Preprocessing)
Based on the analysis above, the `src/preprocessing.py` script applies:
1.  **Cleaning**: Dropping leakage cols, handling dates.
2.  **Imputation**: Filling Median for numericals.
3.  **Feature Eng**: Text Embeddings, TF-IDF, Target Encoding.

We now execute the pipeline to generate `TRAIN.csv` and `TEST.csv`.

In [ ]:
import sys
sys.path.append(os.path.abspath('..'))
from app.src import preprocessing

# Run the full pipeline
preprocessing.main()

## 3.1 Verification
Check that processed data has no missing values.

In [ ]:
train_processed = pd.read_csv(config.TRAIN_DATA_PATH)
print(f"Processed Train Shape: {train_processed.shape}")
print(f"Missing Values Remaining: {train_processed.isnull().sum().sum()}")